# 04 — Build FAISS Index

Embed all 669 fact sentences from Notebook 3 and index them with FAISS so a question gets matched to facts by *meaning*, not keywords — "The Brain" from the original pitch. Rebuilt from the first pass in two ways: it now runs against the expanded 669-fact set, and retrieval returns a confidence score and refuses to force a match when nothing is actually relevant.

In [ ]:
!pip install sentence-transformers faiss-cpu -q

In [ ]:
import pandas as pd
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

DATA_DIR = "../data"
facts_df = pd.read_csv(f"{DATA_DIR}/facts.csv")
print(f"Loaded {len(facts_df)} facts across {facts_df['category'].nunique()} categories")

## Load the embedding model

`all-MiniLM-L6-v2` — free, runs locally, no API key, no per-call cost. It only needs to place similar-*meaning* sentences near each other in vector space; the actual reasoning happens in Notebook 5, with the LLM.

In [ ]:
model = SentenceTransformer("all-MiniLM-L6-v2")
print("Embedding dimension:", model.get_sentence_embedding_dimension())

## Embed every fact and build the index

Normalized vectors + inner-product search = cosine similarity, the standard choice for sentence embeddings.

In [ ]:
embeddings = model.encode(facts_df["text"].tolist(), show_progress_bar=True, convert_to_numpy=True)
embeddings = embeddings.astype("float32")
faiss.normalize_L2(embeddings)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f"Indexed {index.ntotal} facts at dimension {dim}")

In [ ]:
faiss.write_index(index, f"{DATA_DIR}/facts.faiss")
print("Saved index to", f"{DATA_DIR}/facts.faiss")

## Retrieval, with a confidence floor

The first version of this always returned exactly `k` results, even for a question with nothing relevant in the knowledge base — which means the LLM in Notebook 5 would get handed irrelevant context and might try to answer anyway. `MIN_SCORE` fixes that: anything scoring below it gets dropped, and if *nothing* clears the bar, `search()` returns an empty list instead of forcing weak matches through.

**`0.35` below is a starting point, not a verified number** — cosine-similarity scores are specific to this exact model, so look at the real scores in the test below once you run it, and adjust up or down if good matches are getting cut or bad ones are slipping through.

In [ ]:
MIN_SCORE = 0.35

def search(query, k=3, min_score=MIN_SCORE):
    q_vec = model.encode([query], convert_to_numpy=True).astype("float32")
    faiss.normalize_L2(q_vec)
    scores, ids = index.search(q_vec, k)
    results = [
        {"text": facts_df.iloc[i]["text"], "category": facts_df.iloc[i]["category"],
         "n": int(facts_df.iloc[i]["n"]), "score": float(s)}
        for i, s in zip(ids[0], scores[0]) if s >= min_score
    ]
    return results

## Test across a spread of categories

Five questions, deliberately varied: the original geography/salary example, one that needs the education cut, one about the management track, one about AI trust, and one that's completely out of scope on purpose — that last one should come back **empty**, not with three irrelevant facts forced through.

In [ ]:
test_queries = [
    "Does moving to Europe pay more?",
    "Is a CS degree worth it?",
    "Should I become a manager or stay a developer?",
    "Do people trust AI tools?",
    "What is the best pizza topping?",
]

for q in test_queries:
    print(f"QUERY: {q}")
    results = search(q)
    if not results:
        print("  (no confident match — correct behavior for an out-of-scope question)")
    for r in results:
        print(f"  [{r['score']:.2f}] ({r['category']}, n={r['n']}) {r['text'][:90]}")
    print()

## What's better this time

- Runs against all 669 facts, not the original 330
- `search()` now returns category and sample size alongside each fact, not just text — Notebook 5 can use these to decide how much weight to give a result
- A confidence floor means an out-of-scope question can come back empty instead of quietly forcing three irrelevant facts into the LLM's context — the honest failure mode for a grounded system
- Verify the pizza-topping query actually comes back empty when you run this for real, and nudge `MIN_SCORE` if it doesn't

**Next:** Notebook 5 — wire this retrieval function up to an actual LLM and get end-to-end answers working.